In [1]:
import coiled

import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import pytz
import dask
import re
# from io import BytesIO
import requests
import sparse
import time
import zarr
from io import BytesIO
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy

import pygwalker as pyg

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

# coiled notebook start --region=us-east-1

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [3]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [37]:
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=1,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="x8g.xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="x8g.xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                ╷                                                             │
│   Package      │ Note                                                        │
│ ╶──────────────┼───────────────────────────────────────────────────────────╴ │
│   flox         │ Wheel built from ~/flox-0.10.3.tar.gz                       │
│                ╵                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                 ╷                                                ╷           │
│   Package       │ Error                                          │ Level     │
│ ╶───────────────┼────────────────────────────────────────────────┼─────────╴ │
│   pygwalker     │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ segment-analytics-python==2.2.3, but you have  │           │
│                 │ segment-analytics-python 2.2.2.                │           │
│   pydantic_core │ pydantic-core~=2.33.2 has no install candidate │ Warning   │
│                 │ for Python 3.10 linux-aarch64 on conda-forge   │           │
│   awscrt        │ awscrt~=0.26.1 has no install candidate for    │ Warning   │
│                 │ Python 3.10 linux-aarch64 on conda-forge       │           │
│                 ╵                                                ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [ ]:
client.restart() 

In [ ]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

In [ ]:
local_client.shutdown()

In [ ]:
# Conversion of carbon to CO2
C_to_CO2 = 44/12

def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [ ]:
def create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet_name):

    try:
        # Try fetching the file from the S3 URL
        # print(f"Attempting to download file from URL: {spreadsheet}")
        response = requests.get(state_node_lookup_table_s3, timeout=10)
        response.raise_for_status()
        state_node_df = pd.read_excel(BytesIO(response.content), sheet_name=sheet_name)

    except (requests.exceptions.RequestException, Exception) as e:
        print(f"Failed to download file from S3. Falling back to local file. Error: {e}")

        print(f"Reading file from local path: {state_node_lookup_table_local}")
        state_node_df = pd.read_excel(state_node_lookup_table_local, sheet_name=sheet_name)

    return state_node_df

In [ ]:
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

In [ ]:
# Node codes output from model. Covers entire decision tree. Make sure that node codes are right-padded with 0s to 7 digits! 
# Otherwise, only the node codes that are seven digits without 0s will be matched with the node code rasters and output. 
# TODO: I may have accidentally missed some node codes when copying them from the decision tree. Check!
node_codes = np.array([
    1110000, 1120000, 1210000, 1220000, 2111000, 2112000,
    2121100, 2121200, 2122100, 2122200, 2123100, 2123200,
    2124100, 2124200, 2125100, 2125200, 2211100, 2211200, 2212110, 2212120, 
    2212210, 2212220, 2213110, 2213120, 2213210, 2213220,
    2214100, 2214200, 2215100, 2215200, 2221100, 2221200, 
    2222100, 2222200, 2223100, 2223200, 3110000, 3120000, 
    3211111, 3211112, 3211121, 3211122, 3211211, 3211212,
    3211221, 3211222, 3212111, 3212112, 3212121, 3212122,
    3212211, 3212212, 3212221, 3212222, 3221110, 3221120,
    3221210, 3221220, 3222111, 3222112, 3222121, 3222122,
    3222210, 3222220, 4100000, 4210000, 4220000, 4310000,
    4320000, 5100000, 5210000, 5220000, 5310000, 5320000],
dtype=np.uint32)

# # Node codes output from model for 2x2 test area in DRC (23_-5_25_-3)
# node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
#                        2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
#                        2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
#                       dtype=np.uint32)

In [ ]:
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

In [ ]:
# Extracts some metadata/chunk properties to add to the output dataframe
def parse_metadata_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_pixel_yr_\d{4}_\d{4}\.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

In [ ]:
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y':chunk_size}
    ).squeeze()

    return xarray_chunks

In [ ]:
def align_with_nodes(analysis_layer, nodes):
    analysis_layer_sub, nodes_aligned = xr.align(analysis_layer, nodes, join="inner")
    return analysis_layer_sub, nodes_aligned

In [ ]:
def xarray_reduction_sum_count(analysis_layer, node_data):

    reductions = {}

    for func in ["sum", "count"]:
        reduced = xarray_reduce(
            analysis_layer.band_data,
            node_data,
            func=func,
            keep_attrs=True,
            expected_groups=(node_codes),
            reindex=ReindexStrategy(
                blockwise=False,
                array_type=ReindexArrayType.SPARSE_COO
            ),
            fill_value=0
        )

        # Rename variables to reflect reduction type
        if isinstance(reduced, xr.Dataset):
            renamed = reduced.rename({var: f"{var}_{func}" for var in reduced.data_vars})
        else:  # it's a DataArray
            renamed = reduced.rename(f"{reduced.name}_{func}")

        reductions[func] = renamed

    # Merge results: handle Dataset or DataArray combinations
    result = xr.merge([r if isinstance(r, xr.Dataset) else r.to_dataset() for r in reductions.values()])

    return result

In [ ]:
def xarray_reduction(flux_cube, nodes_aligned_data, adm0_data):

    data_cube_by_node = xarray_reduce(
        flux_cube,
        nodes_aligned_data,
        adm0_data,
        func='sum',
        keep_attrs=True,
        expected_groups=(node_codes),
        reindex=ReindexStrategy(
            blockwise=False, array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0   
    )

    return data_cube_by_node

In [ ]:
# Converts flox output to dataframe and does some processing of it
def create_interval_df(coord_dict, state_node_df):

    df = pd.DataFrame(coord_dict)
    # print(df)

    # Replaces numeric values for outputs with names
    df['flux_type'] = df['flux_type'].replace({0: gross_emis_CO2_output_pattern, 1: gross_emis_all_gases_output_pattern, 
                                               2: gross_remv_all_pools_output_pattern, 3: net_flux_output_pattern, 4: "area__ha"})
    # print("with flux_type:", df)
    
    # Classifies the node_codes by larger groupings
    df['node_grp'] = df['state_node'].apply(classify_node)
    # print("with classified nodes:", df)

    # Makes node_codes into strings
    df['state_node'] = 'n' + df['state_node'].astype(str)
    # print("with n prefix:", df)

    # Adds the interval end year to the dataframe
    df['interval_end'] = interval_end_year
    # print("with interval end year:", df)

    df = df.merge(state_node_df[['state_node_with_prefix', 'meaning']],
              left_on='state_node', right_on='state_node_with_prefix',
              how='left')

    # Converts area from m^2 to ha
    df.loc[df['flux_type'].eq('area__ha'), 'value'] = df['value'] / 10000

    # Drop the helper column if you don't want it
    df.drop(columns=['state_node_with_prefix'], inplace=True)
    
    # print(df)

    return df

In [ ]:
# Calculates flux densities (Mg CO2 or CO2e/ha)
# Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682a8c76-0618-800a-a201-18fd404a281f
def calculate_interval_flux_densities(df):

    # Step 1: Filters out area and flux data
    area_df = df[df['flux_type'] == 'area__ha'].copy()
    flux_df = df[df['flux_type'] != 'area__ha'].copy()
    
    # Step 2: Merges flux data with area data on matching keys
    merged = pd.merge(
        flux_df,
        area_df[['state_node', 'gadm_adm0', 'interval_end', 'value']],
        on=['state_node', 'gadm_adm0', 'interval_end'],
        how='left',
        suffixes=('', '_area')
    )
    # print("merged:" merged)
    
    # Step 3: Computes per-hectare flux (converts CO2 to C)
    merged['value_per_ha'] = merged['value'] / merged['value_area'] / C_to_CO2
    
    # Step 4: Prepares flux density rows to append
    new_rows = merged.copy()
    new_rows['flux_type'] = new_rows['flux_type'] + '__C_per_ha'
    new_rows['value'] = new_rows['value_per_ha']
    new_rows = new_rows.drop(columns=['value_area', 'value_per_ha'])
    # print("new rows:", new_rows)
    
    # Step 5: Appends flux density rows to original dataframe
    result_df = pd.concat([df, new_rows], ignore_index=True)

    return result_df

In [ ]:
# Reclassifies state nodes to broad categories
def classify_node(state_node):
    
    node_str = str(state_node)
    first_digit = int(node_str[0])
    # print(first_digit)

    # For broad classes that can be categorized using just the first digit
    one_digit_map = {
        1: 'forest_gain',
        2: 'forest_loss',
        4: 'cropland',
        5: 'grassland'
    }

    # For broad classes that need to be categorized using the first three digits
    three_digit_map = {
        321: 'disturbed_forest',
        322: 'stable_forest'
        # Add more as needed
    }
    
    if first_digit == 3:
        prefix = int(node_str[:3])
        # print(prefix)
        # print(two_digit_map.get(prefix, 'unknown_3x'))
        return three_digit_map.get(prefix, 'unknown_3x')
    else:
        return one_digit_map.get(first_digit, 'unknown')

Code to run zonal stats

In [ ]:
# uri components

# model_version = "version_0_3_2"
# run_date = "20250507"
# chunk_size = 4000

model_version = "version_0_3_3"
run_date = "20250511"
chunk_size = 10000

output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
interval_end_years = [2016]
# interval_end_years = [2020]
# interval_end_years = [2016, 2017, 2018]
# interval_end_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

tile_id = '00N_020E'

# s3 folders for inputs
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/40000_pixels/{run_date}/"

adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"

# Spreadsheet for state_node meanings
state_node_lookup_table_local = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/LULUCF_state_node_lookup_table.xlsx"
state_node_lookup_table_s3 = "http://gfw2-data.s3.amazonaws.com/climate/AFOLU_flux_model/LULUCF/state_node_lookup_tables/LULUCF_state_node_lookup_table.xlsx"
sheet = "v030_20250430"

zarr_s3_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/zarr/{run_date}/"
adm0_zarr_name = f"{zarr_s3_path}global_GADM41_adm0.zarr"
pixel_area_zarr_name = f"{zarr_s3_path}global_pixel_area.zarr"

In [ ]:
print(f"Reading inputs that apply to all intervals: {timestr()}")

adm0_uris = list_folder_uris(adm0_folder)
pixel_area_uris = list_folder_uris(pixel_area_folder)

print("adm0_folder:", adm0_folder)
print(adm0_uris[0])
print(f"Tile count in {adm0_folder}: {len(adm0_uris)}")
print("pixel_area_folder:", pixel_area_folder)
print(pixel_area_uris[0])
print(f"Tile count in {pixel_area_folder}: {len(pixel_area_uris)}")

print(f"   Reading adm0: {timestr()}")
adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
print(f"   Reading pixel_area: {timestr()}")
pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)

print("adm0_xarray_chunks:", adm0_xarray_chunks)
print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

print(f"   zarring adm0: {timestr()}")
adm0_xarray_chunks.to_zarr(adm0_zarr_name, mode='w')
print(f"   zarring pixel area: {timestr()}")
pixel_area_xarray_chunks.to_zarr(pixel_area_zarr_name, mode='w')

In [ ]:
combined_df = pd.DataFrame()
analysis_start_time = time.time()

state_node_df = create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet)
# print(state_node_df)

print("Opening zarrs for non-annual inputs")
adm0_read_zarr = xr.open_zarr(adm0_zarr_name)
pixel_area_read_zarr = xr.open_zarr(pixel_area_zarr_name)


for interval_end_year in interval_end_years:

    interval = f"{interval_end_year-1}_{interval_end_year}"
    # print(interval)

    print(f"Processing {interval}: {timestr()}")
    interval_start_time = time.time()
    
    # Creates a Pandas series of s3 uris for this specific analysis layer
    gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval)
    gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)

    gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval)
    gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
    
    gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval)
    gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
    
    net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval)
    net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
    
    node_folder_interval = node_folder.replace("INTERVAL", interval)
    node_tile_year_uris = list_folder_uris(node_folder_interval)

    print("gross_emis_CO2_folder_interval:", gross_emis_CO2_folder_interval)
    print(gross_emis_CO2_uris[0])
    print(f"Tile count in {gross_emis_CO2_folder_interval}: {len(gross_emis_CO2_uris)}")
    
    print("gross_emis_all_gases_folder_interval:", gross_emis_all_gases_folder_interval)
    print(gross_remv_all_pools_uris[0])
    print(f"Tile count in {gross_emis_all_gases_folder_interval}: {len(gross_remv_all_pools_uris)}")
    
    print("gross_remv_all_pools_folder_interval:", gross_remv_all_pools_folder_interval)
    print(gross_emis_CO2_uris[0])
    print(f"Tile count in {gross_remv_all_pools_folder_interval}: {len(gross_emis_CO2_uris)}")

    print("gross_emis_CO2_folder_interval:", net_flux_all_pools_CO2_folder_interval)
    print(gross_remv_all_pools_uris[0])
    print(f"Tile count in {net_flux_all_pools_CO2_folder_interval}: {len(gross_remv_all_pools_uris)}")

    print("node_folder_interval:", node_folder_interval)
    print(node_tile_year_uris[0])
    print(f"Tile count in {node_folder_interval}: {len(node_tile_year_uris)}")

    # gross_emis_CO2_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/40000_pixels/20250511/{tile_id}__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif'])
    # gross_emis_all_gases_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/40000_pixels/20250511/{tile_id}__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif'])
    # gross_remv_all_pools_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/40000_pixels/20250511/{tile_id}__gross_removals__all_C_pools__MgCO2_pixel_yr_{interval}.tif'])
    # net_flux_all_pools_CO2_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/40000_pixels/20250511/{tile_id}__net_flux__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif'])
    # node_tile_year_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/land_state_node/standard_model/annual_intervals/{interval}/40000_pixels/20250511/{tile_id}__land_state_node_{interval}.tif'])
    
    # Gets input layer metadata, like the output pattern.
    # Note: chunk_id is for the first chunk being processed, not all chunks being processed.
    gross_emis_CO2_output_pattern = parse_metadata_from_uri(gross_emis_CO2_uris)
    gross_emis_all_gases_output_pattern = parse_metadata_from_uri(gross_emis_all_gases_uris)
    gross_remv_all_pools_output_pattern = parse_metadata_from_uri(gross_remv_all_pools_uris)
    net_flux_output_pattern = parse_metadata_from_uri(net_flux_all_pools_CO2_uris)
    node_output_pattern = parse_metadata_from_uri(node_tile_year_uris)
    # print(gross_emis_CO2_output_pattern)
    # print(net_flux_output_pattern)
    
    print(f"   Reading gross emis CO2 only for {interval}: {timestr()}")
    gross_emis_CO2_xarray_chunks = make_xarray_chunks(gross_emis_CO2_uris, chunk_size)
    print(f"   Reading gross emis all gases for {interval}: {timestr()}")
    gross_emis_all_gases_xarray_chunks = make_xarray_chunks(gross_emis_all_gases_uris, chunk_size)   
    print(f"   Reading gross removals for {interval}: {timestr()}")
    gross_remv_all_pools_xarray_chunks = make_xarray_chunks(gross_remv_all_pools_uris, chunk_size)    
    print(f"   Reading net flux CO2 only for {interval}: {timestr()}")
    net_flux_all_pools_CO2_xarray_chunks = make_xarray_chunks(net_flux_all_pools_CO2_uris, chunk_size)   
    print(f"   Reading state_nodes for {interval}: {timestr()}")
    node_xarray_chunks = make_xarray_chunks(node_tile_year_uris, chunk_size)
  
    # print("gross_emis_CO2_xarray_chunks:", gross_emis_CO2_xarray_chunks)
    # print("gross_emis_all_gases_xarray_chunks:", gross_emis_all_gases_xarray_chunks)
    # print("gross_remv_all_pools_xarray_chunks:", gross_remv_all_pools_xarray_chunks)
    # print("net_flux_all_pools_CO2_xarray_chunks:", net_flux_all_pools_CO2_xarray_chunks)
    # print("node_xarray_chunks:", node_xarray_chunks)

    # per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/68309b36-0f48-800a-bd56-67180b55106e
    print(f"   zarring gross emis CO2 only for {interval}: {timestr()}")
    gross_emis_CO2_zarr_name = f"{zarr_s3_path}{interval}/COD_gross_emissions__all_C_pools__CO2_only__MgCO2_{interval}.zarr"
    gross_emis_CO2_xarray_chunks.to_zarr(gross_emis_CO2_zarr_name, mode='w')
    
    print(f"   zarring gross emis all gases only for {interval}: {timestr()}")
    gross_emis_all_gases_zarr_name = f"{zarr_s3_path}{interval}/COD_gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.zarr"
    gross_emis_all_gases_xarray_chunks.to_zarr(gross_emis_all_gases_zarr_name, mode='w')
    
    print(f"   zarring gross removals for {interval}: {timestr()}")
    gross_remv_all_pools_zarr_name = f"{zarr_s3_path}{interval}/COD_gross_removals__all_C_pools__MgCO2_pixel_yr_{interval}.zarr"
    gross_remv_all_pools_xarray_chunks.to_zarr(gross_remv_all_pools_zarr_name, mode='w')
   
    print(f"   zarring net flux CO2 only for {interval}: {timestr()}")
    net_flux_all_pools_CO2_zarr_name = f"{zarr_s3_path}{interval}/COD_net_flux__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.zarr"
    net_flux_all_pools_CO2_xarray_chunks.to_zarr(net_flux_all_pools_CO2_zarr_name, mode='w')
    
    print(f"   zarring nodes for {interval}: {timestr()}")
    node_zarr_name = f"{zarr_s3_path}{interval}/COD_land_state_node_{interval}.zarr"
    node_xarray_chunks.to_zarr(node_zarr_name, mode='w')
    
    print(f"   reading zar for gross emis CO2 only for {interval}: {timestr()}")
    gross_emis_CO2_read_zarr = xr.open_zarr(gross_emis_CO2_zarr_name)
    print(f"   reading zar for gross emis all gases only for {interval}: {timestr()}")
    gross_emis_all_gases_read_zarr = xr.open_zarr(gross_emis_all_gases_zarr_name)
    print(f"   reading zar for gross removals for {interval}: {timestr()}")
    gross_remv_all_pools_read_zarr = xr.open_zarr(gross_remv_all_pools_zarr_name)
    print(f"   reading zar for net flux CO2 only for {interval}: {timestr()}")
    net_flux_all_pools_CO2_read_zarr = xr.open_zarr(net_flux_all_pools_CO2_zarr_name)
    print(f"   reading zar for nodes for {interval}: {timestr()}")
    node_read_zarr = xr.open_zarr(node_zarr_name)

    
    print(f"   Aligning {interval}: {timestr()}")
    gross_emis_CO2_aligned, nodes_aligned = align_with_nodes(gross_emis_CO2_read_zarr, node_read_zarr)
    gross_emis_all_gases_aligned, nodes_aligned = align_with_nodes(gross_emis_all_gases_read_zarr, node_read_zarr)
    gross_remv_all_pools_aligned, nodes_aligned = align_with_nodes(gross_remv_all_pools_read_zarr, node_read_zarr)
    net_flux_all_pools_CO2_aligned, nodes_aligned = align_with_nodes(net_flux_all_pools_CO2_read_zarr, node_read_zarr)

    adm0_aligned, nodes_aligned = align_with_nodes(adm0_read_zarr, node_read_zarr)
    pixel_area_aligned, nodes_aligned = align_with_nodes(pixel_area_read_zarr, node_read_zarr)
    
#     # print("gross_emis_CO2_aligned:", gross_emis_CO2_aligned)
#     # print("gross_emis_all_gases_aligned:", gross_emis_all_gases_aligned)
#     # print("gross_remv_all_pools_aligned:", gross_remv_all_pools_aligned)
#     # print("net_flux_all_pools_CO2_aligned:", net_flux_all_pools_CO2_aligned)
#     # print("nodes_aligned:", nodes_aligned)
#     # print("adm0_aligned:", adm0_aligned)
#     # print("pixel_area_aligned:", pixel_area_aligned)
    
    nodes_aligned_data = nodes_aligned.band_data
    nodes_aligned_data.name = 'state_node'

    adm0_data = adm0_aligned.band_data
    adm0_data.name = 'gadm_adm0'

    print(f"   Stacking {interval}: {timestr()}")
    flux_cube = xr.DataArray(dask.array.stack((gross_emis_CO2_aligned.band_data, 
                                               gross_emis_all_gases_aligned.band_data, 
                                               gross_remv_all_pools_aligned.band_data, 
                                               net_flux_all_pools_CO2_aligned.band_data, 
                                               pixel_area_aligned.band_data)), 
                             dims=('flux_type', 'y', 'x'))
    # print(flux_cube)


    print(f"   Reducing {interval}: {timestr()}")

    # Analysis for node_codes and adm0
    data_cube_by_contexts = xarray_reduce(
        flux_cube,
        *(nodes_aligned_data, adm0_data),
        func='sum',
        expected_groups=(node_codes, gadm_adm0_ids),
        reindex=ReindexStrategy(
            blockwise=False, array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0
    )

    print(f"   Computing {interval}: {timestr()}")
    result = data_cube_by_contexts.compute()

    print(f"   Processing output for {interval}: {timestr()}")
    sparse_data = result.data

    dim_names = result.dims
    indices = sparse_data.coords
    values = sparse_data.data
    
    coord_dict = {
        dim: result.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    coord_dict["value"] = values

    # Creates the dataframe for the interval and does some processing of it
    df = create_interval_df(coord_dict, state_node_df)
    # print(df)

    df = calculate_interval_flux_densities(df)
    # print(df)

    combined_df = pd.concat([combined_df, df])

    interval_end_time = time.time()
    print(f"   {interval} took {round(interval_end_time - interval_start_time)} seconds")

combined_df = combined_df.reset_index(drop=True)

analysis_end_time = time.time()
print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")

print(combined_df)

In [ ]:
# combined_df[(combined_df.flux_type == gross_remv_all_pools_output_pattern) 
# & (combined_df.gadm_adm0 == 180)]
combined_df[(combined_df.flux_type == 'area__ha') 
& (combined_df.state_node == 'n1110000')]

In [ ]:
combined_df[(combined_df.flux_type == net_flux_output_pattern) 
& (combined_df.state_node == 'n1110000')]
# combined_df[(combined_df.state_node == 'n2112000')]
# combined_df.groupby(combined_df.gadm_adm0).sum()

In [ ]:
combined_df[(combined_df.state_node == 'n1110000')]

In [ ]:
result_df[(result_df.flux_type == f'{gross_remv_all_pools_output_pattern}__per_ha') 
& (result_df.state_node == 'n5100000')]

In [ ]:
walker = pyg.walk(combined_df)

In [ ]:
combined_df_wide = combined_df.pivot(index=['state_node', 'interval_end', 'node_grp', 'gadm_adm0', 'meaning'], columns="flux_type", values="value").reset_index()
combined_df_wide

In [ ]:
walker = pyg.walk(combined_df_wide)

In [ ]:
vis_spec = r"""{"config":[{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_jRXC","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_Lh92","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_VaYi","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_Mk45","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_cqa-","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_TkdT","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_oZRA","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_O_zx","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_n1-c","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_wRjq","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_oJki","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_vdme","fid":"gross_removals__all_C_pools__MgCO2__per_ha","name":"gross_removals__all_C_pools__MgCO2__per_ha","basename":"gross_removals__all_C_pools__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_mzLG","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GaeA","fid":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","dragId":"GW_61VCEMWD"},{"fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","dragId":"GW_UDav70fB"},{"fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","dragId":"GW_hXBRtcNi"},{"fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","dragId":"GW_Iy0Mvc5p"}],"rows":[{"dragId":"gw_7UP-","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_ZSNW","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_vvjf","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"quantitative","analyticType":"dimension"}],"color":[{"dragId":"gw_OdKn","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_SA4H","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_ra-I","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_Z3eu","name":"Emissions"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_jRXC","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_Lh92","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_VaYi","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_Mk45","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_cqa-","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_TkdT","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_oZRA","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_O_zx","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_n1-c","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_wRjq","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_oJki","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_vdme","fid":"gross_removals__all_C_pools__MgCO2__per_ha","name":"gross_removals__all_C_pools__MgCO2__per_ha","basename":"gross_removals__all_C_pools__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_mzLG","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GaeA","fid":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","dragId":"GW_6oskDy5X"},{"fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","dragId":"GW_4TQPC6VF"},{"fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","dragId":"GW_ZSt4HTqn"},{"fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","dragId":"GW_kcs5yn67"}],"rows":[{"dragId":"gw_hjMZ","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","dragId":"gw_QA6y"}],"columns":[{"dragId":"gw_OdEp","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"quantitative","analyticType":"dimension"}],"color":[{"dragId":"gw_mRsu","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_SA4H","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_ra-I","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_sADd","name":"Emission factor"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_jRXC","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_Lh92","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_VaYi","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_Mk45","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_cqa-","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_TkdT","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_oZRA","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_O_zx","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_n1-c","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_wRjq","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_oJki","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_vdme","fid":"gross_removals__all_C_pools__MgCO2__per_ha","name":"gross_removals__all_C_pools__MgCO2__per_ha","basename":"gross_removals__all_C_pools__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_mzLG","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GaeA","fid":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","dragId":"GW_XT9f8Juz"},{"fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","dragId":"GW_OYc2IgEP"},{"fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","dragId":"GW_XOqMcQby"},{"fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","semanticType":"quantitative","analyticType":"measure","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","dragId":"GW_GsHnSMdO"}],"rows":[{"dragId":"gw_7UP-","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_CeNn","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_vvjf","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"quantitative","analyticType":"dimension"}],"color":[{"dragId":"gw_AnRV","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_SA4H","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_ra-I","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_2-Pi","name":"Emission area"}],"chart_map":{},"workflow_list":[{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","gadm_adm0","meaning"],"measures":[{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"raw","fields":["interval_end","gadm_adm0","meaning","gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha"]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","gadm_adm0","meaning"],"measures":[{"field":"area__ha","agg":"sum","asFieldKey":"area__ha_sum"}]}]}]}],"timezoneOffsetSeconds":-14400,"version":"0.3.17"}"""
pyg.walk(combined_df_wide, spec=vis_spec)